<a href="https://colab.research.google.com/github/marina-popova11/MediaImpactOnCryptoPrices/blob/training/train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import DatasetDict, Dataset, load_from_disk
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

import wandb
wandb.init(mode="offline")

def load_arrow(filepath):
    try:
        dataset = load_from_disk(filepath)
        print("Successfully loaded dataset from disk")
        return dataset
    except Exception as e:
        print(e)
        return None

dataset_path = "/content/drive/MyDrive/crypto_arrow_dataset"

import os
if os.path.exists(dataset_path):
    print(f"Path exists: {os.path.exists(dataset_path)}")
    print(f"Files in directory: {os.listdir(dataset_path)}")
else:
    print(f"Path does not exist. Current directory: {os.getcwd()}")
    print(f"Available files: {os.listdir('.')}")

dataset = load_arrow(dataset_path)

if dataset is None:
    try:
        dataset = DatasetDict({
            'train': load_from_disk(f"{dataset_path}/train"),
            'test': load_from_disk(f"{dataset_path}/test"),
            'validation': load_from_disk(f"{dataset_path}/validation")
        })
        print("dataset download throw DatasetDict")
    except Exception as e:
        print(e)
        exit()

print(f"\nDataset type: {type(dataset)}")
print(f"Dataset keys: {list(dataset.keys())}")

available_columns = dataset['train'].column_names
text_column = "BODY"
label_column = "label"
if text_column not in available_columns:
    text_column = available_columns[1]
    print(f" 'text' not found in available columns, use: {text_column}")

if label_column not in available_columns:
    label_cand = [col for col in available_columns if any(keyword in col.lower() for keyword in ['label', 'sentiment', 'class', 'category'])]
    if label_cand:
        label_column = label_cand[0]
        print(f"Label column:'{label_column}")
    else:
        print("Label column not found")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples[text_column],
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

tokenized_datasets = DatasetDict()
for name, data in dataset.items():
    tokenized_datasets[name] = data.map(
        tokenize_function,
        batched=True,
        batch_size=1000,
    )

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Path exists: True
Files in directory: ['test', 'validation', 'train', 'dataset_dict.json']
Successfully loaded dataset from disk

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>
Dataset keys: ['train', 'validation', 'test']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
columns_to_keep = ["input_ids", "attention_mask", "label"]
for name in tokenized_datasets.keys():
    tokenized_datasets[name].set_format(
        type="torch",
        columns=columns_to_keep
    )

print(tokenized_datasets["train"]["label"][:5])
print(type(tokenized_datasets["train"]["label"][0]))

tensor([2, 2, 2, 1, 2])
<class 'torch.Tensor'>


In [ ]:
!cp -r "/content/drive/MyDrive/bert_encoder_for_volatility/checkpoint-1400" "./bert_encoder_for_volatility/"

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

training_args = TrainingArguments(
    output_dir="./bert_encoder_for_volatility",
    overwrite_output_dir=False,
    learning_rate=3e-5,
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=None,
    save_total_limit=5
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions, average='weighted')
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

train_result = trainer.train()
trainer.save_model("./fine_tuned_bert_encoder")
encoder = model.distilbert
encoder.save_pretrained("./fine_tuned_bert_encoder_only")
tokenizer.save_pretrained("./fine_tuned_bert_encoder_only")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1268217721.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,F1
500,0.560500,0.520091,0.777505,0.772694
1000,0.499500,0.508027,0.784529,0.780820
1500,0.478300,0.463318,0.798753,0.799597
2000,0.488900,0.448584,0.808499,0.809016
2500,0.456200,0.445864,0.806743,0.806741


('./fine_tuned_bert_encoder_only/tokenizer_config.json',
 './fine_tuned_bert_encoder_only/special_tokens_map.json',
 './fine_tuned_bert_encoder_only/vocab.txt',
 './fine_tuned_bert_encoder_only/added_tokens.json',
 './fine_tuned_bert_encoder_only/tokenizer.json')

In [ ]:
test_result = trainer.evaluate(tokenized_datasets["test"])
print(f"Test Accuracy: {test_result.get('eval_accuracy', 0):.4f}")
print(f"Test F1: {test_result.get('eval_f1', 0):.4f}")
print(f"Test Loss: {test_result.get('eval_loss', 0):.4f}")

Test Accuracy: 0.8112
Test F1: 0.8110
Test Loss: 0.4444


In [6]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset
from torch.utils.data import Dataset, DataLoader
from datasets import DatasetDict, Dataset, load_from_disk
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel, DistilBertForSequenceClassification, DistilBertTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32

def load_arrow(filepath):
    try:
        dataset = load_from_disk(filepath)
        print("Successfully loaded dataset from disk")
        return dataset
    except Exception as e:
        print(e)
        return None

dataset_path = "/content/drive/MyDrive/crypto_arrow_dataset"

import os
if os.path.exists(dataset_path):
    print(f"Path exists: {os.path.exists(dataset_path)}")
    print(f"Files in directory: {os.listdir(dataset_path)}")
else:
    print(f"Path does not exist. Current directory: {os.getcwd()}")
    print(f"Available files: {os.listdir('.')}")

text_column = "BODY"
label_column = "label"
dataset = load_arrow(dataset_path)
test_df = dataset['test']
texts = [str(x) for x in test_df[text_column]]
labels = test_df[label_column]
model_path = "/content/drive/MyDrive/bert_final_81_percent"

tokenizer = DistilBertTokenizer.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(model_path)
model.eval()
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

dataset = torch.utils.data.TensorDataset(
    encodings["input_ids"],
    encodings["attention_mask"],
    torch.tensor(labels)
)

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids, attention_mask, batch_labels = batch

        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)

print(f"Accuracy: {accuracy:.4f}")
print(classification_report(all_labels, all_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Path exists: True
Files in directory: ['test', 'validation', 'train', 'dataset_dict.json']
Successfully loaded dataset from disk


Evaluating: 100%|██████████| 356/356 [01:25<00:00,  4.15it/s]

Accuracy: 0.8245
              precision    recall  f1-score   support

           0       0.83      0.85      0.84      2636
           1       0.73      0.71      0.72      3384
           2       0.88      0.88      0.88      5369

    accuracy                           0.82     11389
   macro avg       0.81      0.82      0.81     11389
weighted avg       0.82      0.82      0.82     11389


Confusion Matrix:
[[2253  314   69]
 [ 379 2407  598]
 [  80  559 4730]]
